In [10]:
import json
import re
import numpy as np

from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

In [3]:
CHUNK_PATH = "../data/processed/wk10_chunks.json"

with open(CHUNK_PATH, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print("Chunks Loaded:", len(chunks))

Chunks Loaded: 48


In [4]:
def preprocess(text):

    text = text.lower()

    text = re.sub(r'[^a-zA-Z0-9 ]', '', text)

    return text.split()

In [5]:
bm25_corpus = []

for chunk in chunks:

    tokens = preprocess(chunk["text"])

    bm25_corpus.append(tokens)

bm25 = BM25Okapi(bm25_corpus)

print("BM25 Ready")

BM25 Ready


In [6]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma(

    persist_directory="../chroma_db",
    embedding_function=embedding_model
)

print("Chroma Loaded")

c:\Python310\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\himkar vashistha\AppData\Local\Temp\ipykernel_42852\2961996672.py:5: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Chroma Loaded


In [11]:
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

c:\Python310\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
c:\Python310\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\himkar vashistha\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In orde

In [12]:
def hybrid_search(query, top_k=5):

    # ---------- BM25 ----------
    query_tokens = preprocess(query)

    bm25_scores = bm25.get_scores(query_tokens)

    bm25_top_indices = np.argsort(bm25_scores)[::-1][:10]

    bm25_results = []

    for idx in bm25_top_indices:

        bm25_results.append({
            "text": chunks[idx]["text"],
            "page": chunks[idx]["page"],
            "content_type": chunks[idx]["content_type"],
            "score": float(bm25_scores[idx]),
            "source": "bm25"
        })

    # ---------- Semantic ----------
    semantic_docs = vectorstore.similarity_search_with_score(query, k=10)

    semantic_results = []

    for doc, score in semantic_docs:

        semantic_results.append({
            "text": doc.page_content,
            "page": doc.metadata["page"],
            "content_type": doc.metadata["content_type"],
            "score": float(score),
            "source": "semantic"
        })

    # ---------- Fusion ----------
    combined_results = bm25_results + semantic_results

    unique_chunks = {}

    for item in combined_results:

        key = item["text"][:150]

        if key not in unique_chunks:
            unique_chunks[key] = item

    candidate_chunks = list(unique_chunks.values())

    # ---------- Cross Encoder Reranking ----------
    pairs = []

    for item in candidate_chunks:

        pairs.append((query, item["text"]))

    rerank_scores = reranker.predict(pairs)

    for i, score in enumerate(rerank_scores):

        candidate_chunks[i]["rerank_score"] = float(score)

    # ---------- Final Sorting ----------
    candidate_chunks = sorted(
        candidate_chunks,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    return candidate_chunks[:top_k]

In [13]:
query = "Why does an object resist motion?"

results = hybrid_search(query)

In [15]:
for r in results:

    print("="*60)

    print("Source:", r["source"])
    print("Page:", r["page"])
    print("Type:", r["content_type"])

    print(r["text"][:500])

Source: bm25
Page: 10
Type: concept
. Skateboards are not cartwheels to observe the motion of the two carts when the children throw the as effective because it is difficult to maintain bag towards each other. straight-line motion. What you have learnt • First law of motion: An object continues to be in a state of rest or of uniform motion along a straight line unless acted upon by an unbalanced force. • The natural tendency of objects to resist a change in their state of rest or of uniform motion is called inertia. • The mass of a
Source: bm25
Page: 5
Type: worked_example
. Clearly, heavier or more massive objects offer • The inertia of the coin tries to larger inertia. Quantitatively, the inertia of an maintain its state of rest even when object is measured by its mass. We may thus the card flows off. relate inertia and mass as follows: Inertia is the natural tendency of an object to resist a change in its state of motion or of rest. The mass of an object is a measure of its inertia. 